# 1.2. Tải nguồn Đa Phiên bản, Lọc Tệp Nhị phân và Kiểm toán Độc lập

## 1. Cơ chế tải luồng bền vững, HTTP `Range` Resume và lọc ngay trên Tar Stream

- **Khóa miền chuyên dụng & Quản trị nhịp mạng (`source_downloader.py`, `rate_limiter.py`)**: Mọi yêu cầu tải gói nguồn đều đi qua `https://export.arxiv.org/src/<id>vN` với `Accept-Encoding: identity` (tránh lỗi giải nén kép `gzip-on-gzip`), khóa đơn luồng `LocalMutex` đảm bảo giãn cách **`≥ 3,0 giây` tính từ thời điểm hoàn tất truyền tải** và tôn trọng tuyệt đối header `Retry-After`.
- **Tiếp tục tải dở dang qua HTTP `Range` (`206 Partial Content`)**: Các gói lớn bị ngắt kết nối giữa chừng được lưu vào tệp `.part`, kiểm tra khớp `Content-Range` để tải nối tiếp mà không tốn lại băng thông; các gói nặng `≥ 20 MiB` lỗi lặp được hoãn về cuối phiên (*heavy-archive deferral*) và toàn bộ tiến độ được đóng gói gia tăng mỗi 25 bài lên chuỗi sao lưu `1.713` gói kiểm chứng băm SHA-256 (`drive_backup.py`).
- **Lọc bỏ hình ảnh ngay trong luồng giải nén (`_safe_extract_tar`)**: Thay vì giải nén toàn bộ ra đĩa rồi xóa, trình đọc duyệt trực tiếp từng entry của luồng `.tar.gz`, cộng kích thước gốc vào `size_before_bytes`, chặn Path Traversal/Tar Bomb và **chỉ ghi xuống đĩa các tệp văn bản thuộc danh sách cho phép `{.tex, .bib, .bbl}`** (giữ `.bbl` theo hướng dẫn QA giảng viên), đồng thời **giữ nguyên vẹn các lệnh `\includegraphics` và môi trường `figure`/`table` bên trong tệp `.tex`** phục vụ Milestone 2.

## 2. Kết quả kiểm toán nguồn trên 41.498 phiên bản và hiệu quả giảm dung lượng (Bytes & GiB)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
sa = m1["source_audit"]
vs = m1["version_status"]
audit_rows = [
    ("Có mã nguồn TeX xác minh (tex_verified)", sa["tex_verified"], 100.0 * sa["tex_verified"] / scope["versions"], "Đủ .tex/.bib/.bbl sạch để đưa vào Parser M2"),
    ("Không có TeX: chỉ nộp PDF (pdf_only)", vs["pdf_only"], 100.0 * vs["pdf_only"] / scope["versions"], "Ngắt sớm ở 16 byte đầu %PDF-; bảo toàn bản ghi empty_verified"),
    ("Không có TeX: gói nguồn thiếu .tex (no_latex)", vs["no_latex"], 100.0 * vs["no_latex"] / scope["versions"], "Xác minh hợp lệ thuộc nhóm empty_verified (2.756 + 8 = 2.764)"),
    ("Máy chủ arXiv trả HTTP 404 (not_found_verified)", sa["not_found_verified"], 100.0 * sa["not_found_verified"] / scope["versions"], "Nguồn bị tác giả rút hoặc ẩn vĩnh viễn trên arXiv"),
    ("Còn chờ / lỗi tải nguồn (source_pending_or_failed)", sa["source_pending_or_failed"], 100.0 * sa["source_pending_or_failed"] / scope["versions"], "11 phiên bản cuối phiên từ phân vùng TRIET"),
]
df_aud = pd.DataFrame(audit_rows, columns=["Phân loại kiểm toán phiên bản", "Số phiên bản", "Tỷ lệ (%)", "Ý nghĩa kiểm toán"])
display(style_table(
    df_aud,
    caption="Bảng 2.1. Kết quả kiểm toán độc lập trên toàn bộ 41.498 phiên bản (xác minh 41.487 / 41.498 = 99,97%).",
    int_cols=["Số phiên bản"],
    pct_cols=["Tỷ lệ (%)"],
    bar_cols={"Số phiên bản": ("var(--ah-bar-fill)", None)},
))

# Bảng kích thước bài báo bằng BYTES bắt buộc theo đúng đặc tả M1 + quy đổi MiB/GiB
tot_before_b = m1["total_source_size_before_bytes"]
tot_after_b = m1["total_source_size_after_bytes"]
saved_b = tot_before_b - tot_after_b
df_size = pd.DataFrame([
    ("Kích thước TB 1 bài trước lọc ảnh (22.748 bài đủ TeX)", f"{fmt_num(m1['mean_before_bytes'], 2)} byte", f"{fmt_num(m1['mean_before_bytes']/1048576, 3)} MiB/bài"),
    ("Kích thước TB 1 bài sau lọc chỉ giữ .tex/.bib/.bbl", f"{fmt_num(m1['mean_after_bytes'], 2)} byte", f"{fmt_num(m1['mean_after_bytes']/1048576, 3)} MiB/bài"),
    ("Kích thước TB tính trên toàn tập 25.000 bài (trước → sau)", f"{fmt_num(tot_before_b/scope['papers'], 2)} → {fmt_num(tot_after_b/scope['papers'], 2)} byte", f"{fmt_num(tot_before_b/scope['papers']/1048576, 3)} → {fmt_num(tot_after_b/scope['papers']/1048576, 3)} MiB/bài"),
    ("Tổng dung lượng toàn corpus 25.000 bài trước lọc", f"{fmt_int(tot_before_b)} byte", f"{fmt_num(tot_before_b/1024**3, 2)} GiB ({fmt_num(tot_before_b/1e9, 2)} GB)"),
    ("Tổng dung lượng toàn corpus 25.000 bài sau lọc", f"{fmt_int(tot_after_b)} byte", f"{fmt_num(tot_after_b/1024**3, 2)} GiB ({fmt_num(tot_after_b/1e9, 2)} GB)"),
    ("Dung lượng nhị phân tiết kiệm được (Tỷ lệ giảm)", f"{fmt_int(saved_b)} byte", f"{fmt_num(saved_b/1024**3, 2)} GiB (-{fmt_pct(m1['source_reduction'], 2, True)})"),
], columns=["Chỉ tiêu dung lượng mã nguồn", "Giá trị chính xác (Bytes)", "Quy đổi nhị phân (MiB / GiB)"])
display(style_table(df_size, caption="Bảng 2.2. Kích thước trung bình bài báo (tính bằng byte) và tổng dung lượng trước/sau khi lọc bỏ hình ảnh."))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.0), gridspec_kw={"width_ratios": [1.25, 1.0]})
cols_a = [BOOK_PALETTE["emerald"], BOOK_PALETTE["slate"], "#94A3B8", BOOK_PALETTE["amber"], BOOK_PALETTE["crimson"]]
y_a = np.arange(len(df_aud))
ax1.barh(y_a, df_aud["Số phiên bản"], color=cols_a, height=0.56)
ax1.set_yticks(y_a); ax1.set_yticklabels(["TeX xác minh", "PDF-only", "Không có .tex", "HTTP 404", "Chờ/lỗi nguồn"])
ax1.invert_yaxis(); ax1.set_xscale("symlog", linthresh=30); ax1.set_xlim(0, 120000); ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản (thang symlog)"); ax1.set_title("Hình 2.1a. Kiểm toán 41.498 phiên bản nguồn")
for i_a, r_a in enumerate(df_aud.to_dict("records")):
    ax1.text(r_a["Số phiên bản"] * 1.16 + 2, i_a, f"{fmt_int(r_a['Số phiên bản'])} ({fmt_pct(r_a['Tỷ lệ (%)'])})",
             va="center", fontsize=8.1, fontweight="bold")

bars_s = ax2.bar(["Trước lọc ảnh\n(gốc tar/gzip)", "Sau lọc giữ\n.tex / .bib / .bbl"],
                 [m1["mean_before_bytes"] / 1e6, m1["mean_after_bytes"] / 1e6],
                 color=[BOOK_PALETTE["slate"], BOOK_PALETTE["teal"]], width=0.52)
ax2.set_ylabel("Triệu byte / bài báo (MB)"); ax2.set_ylim(0, 12.2); ax2.grid(axis="y")
ax2.set_title("Hình 2.1b. Giảm 89,30% kích thước TB bài báo")
ax2.text(bars_s[0].get_x() + bars_s[0].get_width()/2, 10.2,
         f"{fmt_int(m1['mean_before_bytes'])} B\n(9,484 MiB)", ha="center", fontsize=8.3, fontweight="bold")
ax2.text(bars_s[1].get_x() + bars_s[1].get_width()/2, 1.4,
         f"{fmt_int(m1['mean_after_bytes'])} B\n(1,015 MiB • -89,30%)",
         ha="center", fontsize=8.3, fontweight="bold", color=BOOK_PALETTE["teal"])
plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/source_downloader.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/source_downloader.py).
- Mã nguồn triển khai: [`src/crawler/retry_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/retry_sources.py).
- Mã nguồn triển khai: [`src/crawler/validate_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/validate_sources.py).
- Mã nguồn triển khai: [`src/crawler/drive_backup.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/drive_backup.py).